# Retrieval evaluation from saved outputs

This notebook only evaluates saved retrieval outputs.

It does not run BM25, dense retrieval, hybrid retrieval, Qdrant search, or reranking. It reads the JSONL files produced by the retrieval output notebook, filters `SUPPORTED` and `REFUTED` claims, and computes retrieval metrics.

# Workflow

Use this notebook after running the retrieval output notebook.

On the dev split, use this notebook to compare hybrid alphas and choose the best alpha.

On the test split, set `BEST_ALPHA_FROM_DEV` to the alpha selected on dev, then report the final retrieval results for BM25, Dense, Best Hybrid, and Best Hybrid plus Reranker.

# Configuration

In [1]:
import json
import math
import re
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd

MODE = "dev"  # "dev" or "test"

INPUT_ROOT = Path("/content/retrieval_outputs")
INPUT_DIR = INPUT_ROOT / MODE

OUTPUT_ROOT = Path("/content/retrieval_evaluation")
OUTPUT_DIR = OUTPUT_ROOT / MODE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_FIELD = "label"
GOLD_EVIDENCE_FIELD = "evidence_ids"
EVAL_LABELS = {"SUPPORTED", "REFUTED"}

K_VALUES = [1, 3, 5, 10]

# Set this after dev evaluation.
# This value is used to build the compact final table, especially on the test split.
BEST_ALPHA_FROM_DEV = 0.5

def alpha_tag(alpha: float) -> str:
    return f"alpha{int(round(alpha * 10)):02d}"

print(f"Mode: {MODE}")
print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

Mode: dev
Input directory: /content/retrieval_outputs/dev
Output directory: /content/retrieval_evaluation/dev


# Load retrieval output files

In [2]:
def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(
                    f"Invalid JSONL file: {path}\n"
                    f"Line: {line_number}\n"
                    f"Error: {e}\n"
                    f"Line preview: {line[:500]}"
                ) from e
    return rows


expected_files = [
    ("BM25 top-20", INPUT_DIR / f"bm25_top20_all_labels_{MODE}.jsonl"),
    ("Dense top-20", INPUT_DIR / f"dense_top20_all_labels_{MODE}.jsonl"),
]

if MODE == "dev":
    for alpha in [0.3, 0.5, 0.7]:
        tag = alpha_tag(alpha)
        expected_files.append((
            f"Hybrid {tag} top-20",
            INPUT_DIR / f"hybrid_{tag}_top20_all_labels_{MODE}.jsonl",
        ))

    # File 1 hiện tại của bạn đang tạo reranker alpha05 trên dev.
    # Nếu sau này bạn chạy reranker alpha khác thì sửa RERANK_ALPHA_FOR_EVAL.
    RERANK_ALPHA_FOR_EVAL = 0.5
    rerank_tag = alpha_tag(RERANK_ALPHA_FOR_EVAL)

    expected_files.append((
        f"Hybrid reranker {rerank_tag} top-10",
        INPUT_DIR / f"hybrid_reranker_{rerank_tag}_top10_all_labels_{MODE}.jsonl",
    ))

elif MODE == "test":
    best_tag = alpha_tag(BEST_ALPHA_FROM_DEV)

    expected_files.append((
        f"Hybrid {best_tag} top-20",
        INPUT_DIR / f"hybrid_{best_tag}_top20_all_labels_{MODE}.jsonl",
    ))

    expected_files.append((
        f"Hybrid reranker {best_tag} top-10",
        INPUT_DIR / f"hybrid_reranker_{best_tag}_top10_all_labels_{MODE}.jsonl",
    ))

else:
    raise ValueError("MODE must be 'dev' or 'test'.")


missing_files = [str(path) for _, path in expected_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required retrieval output files:\n" + "\n".join(missing_files)
    )

candidate_files = expected_files

print("Evaluation files:")
for method_name, path in candidate_files:
    print(f"{method_name}: {path}")

print("Number of evaluation files:", len(candidate_files))

Evaluation files:
BM25 top-20: /content/retrieval_outputs/dev/bm25_top20_all_labels_dev.jsonl
Dense top-20: /content/retrieval_outputs/dev/dense_top20_all_labels_dev.jsonl
Hybrid alpha03 top-20: /content/retrieval_outputs/dev/hybrid_alpha03_top20_all_labels_dev.jsonl
Hybrid alpha05 top-20: /content/retrieval_outputs/dev/hybrid_alpha05_top20_all_labels_dev.jsonl
Hybrid alpha07 top-20: /content/retrieval_outputs/dev/hybrid_alpha07_top20_all_labels_dev.jsonl
Hybrid reranker alpha05 top-10: /content/retrieval_outputs/dev/hybrid_reranker_alpha05_top10_all_labels_dev.jsonl
Number of evaluation files: 6


# Metric computation

In [3]:
def compute_metrics_for_rows(rows: List[Dict[str, Any]], method_name: str):
    eval_rows = [
        row for row in rows
        if row[LABEL_FIELD] in EVAL_LABELS
    ]

    per_claim = []

    metric_values = {f"Recall@{k}": [] for k in K_VALUES}
    metric_values.update({f"Hit@{k}": [] for k in K_VALUES})
    metric_values.update({f"nDCG@{k}": [] for k in K_VALUES})
    metric_values["MRR@10"] = []

    for row in eval_rows:
        gold_ids = {str(cid) for cid in row[GOLD_EVIDENCE_FIELD]}
        retrieved_ids = [str(item["chunk_id"]) for item in row["retrieved_evidence"]]

        first_relevant_rank = None
        for index, cid in enumerate(retrieved_ids[:10], start=1):
            if cid in gold_ids:
                first_relevant_rank = index
                break

        mrr = 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank
        metric_values["MRR@10"].append(mrr)

        claim_result = {
            "method": method_name,
            "claim_id": row.get("claim_id"),
            "label": row[LABEL_FIELD],
            "gold_count": len(gold_ids),
            "first_relevant_rank": first_relevant_rank,
            "MRR@10": mrr,
        }

        for k in K_VALUES:
            top_k = retrieved_ids[:k]
            relevant_count = len(gold_ids.intersection(top_k))

            recall = relevant_count / len(gold_ids) if gold_ids else 0.0
            hit = 1.0 if relevant_count > 0 else 0.0

            dcg = 0.0
            for rank_index, cid in enumerate(top_k, start=1):
                if cid in gold_ids:
                    dcg += 1.0 / math.log2(rank_index + 1)

            ideal_relevant = min(len(gold_ids), k)
            idcg = sum(
                1.0 / math.log2(rank_index + 1)
                for rank_index in range(1, ideal_relevant + 1)
            )
            ndcg = dcg / idcg if idcg > 0 else 0.0

            metric_values[f"Recall@{k}"].append(recall)
            metric_values[f"Hit@{k}"].append(hit)
            metric_values[f"nDCG@{k}"].append(ndcg)

            claim_result[f"Recall@{k}"] = recall
            claim_result[f"Hit@{k}"] = hit
            claim_result[f"nDCG@{k}"] = ndcg

        per_claim.append(claim_result)

    summary = {
        "method": method_name,
        "mode": MODE,
        "num_claims": len(eval_rows),
    }

    for metric_name, values in metric_values.items():
        summary[metric_name] = float(np.mean(values)) if values else 0.0

    return summary, per_claim

# Evaluate all available retrieval outputs

In [4]:
summary_rows = []
per_claim_rows = []

for method_name, path in candidate_files:
    rows = read_jsonl(path)
    summary, per_claim = compute_metrics_for_rows(rows, method_name)
    summary_rows.append(summary)
    per_claim_rows.extend(per_claim)

metrics_df = pd.DataFrame(summary_rows)
per_claim_df = pd.DataFrame(per_claim_rows)

preferred_columns = [
    "method",
    "mode",
    "num_claims",
    "Recall@1",
    "Recall@3",
    "Recall@5",
    "Recall@10",
    "Hit@1",
    "Hit@3",
    "Hit@5",
    "Hit@10",
    "MRR@10",
    "nDCG@1",
    "nDCG@3",
    "nDCG@5",
    "nDCG@10",
]

metrics_df = metrics_df[[col for col in preferred_columns if col in metrics_df.columns]]

metrics_path = OUTPUT_DIR / f"retrieval_metrics_{MODE}.csv"
metrics_json_path = OUTPUT_DIR / f"retrieval_metrics_{MODE}.json"
per_claim_path = OUTPUT_DIR / f"retrieval_per_claim_{MODE}.csv"
per_claim_jsonl_path = OUTPUT_DIR / f"retrieval_per_claim_{MODE}.jsonl"

metrics_df.to_csv(metrics_path, index=False)
metrics_df.to_json(metrics_json_path, orient="records", force_ascii=False, indent=2)
per_claim_df.to_csv(per_claim_path, index=False)

with open(per_claim_jsonl_path, "w", encoding="utf-8") as f:
    for row in per_claim_df.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

display(metrics_df)

print(f"Saved metrics: {metrics_path}")
print(f"Saved per-claim results: {per_claim_path}")

,method,mode,num_claims,Recall@1,Recall@3,Recall@5,Recall@10,Hit@1,Hit@3,Hit@5,Hit@10,MRR@10,nDCG@1,nDCG@3,nDCG@5,nDCG@10
0,BM25 top-20,dev,176,0.732955,0.812500,0.840909,0.863636,0.732955,0.812500,0.840909,0.863636,0.777264,0.732955,0.779423,0.790911,0.798249
1,Dense top-20,dev,176,0.670455,0.767045,0.818182,0.880682,0.670455,0.767045,0.818182,0.880682,0.729796,0.670455,0.724701,0.745231,0.765599
2,Hybrid alpha03 top-20,dev,176,0.767045,0.840909,0.875000,0.875000,0.767045,0.840909,0.875000,0.875000,0.809375,0.767045,0.811416,0.825850,0.825850
3,Hybrid alpha05 top-20,dev,176,0.772727,0.863636,0.903409,0.926136,0.772727,0.863636,0.903409,0.926136,0.825110,0.772727,0.825621,0.842252,0.849803
4,Hybrid alpha07 top-20,dev,176,0.789773,0.857955,0.886364,0.914773,0.789773,0.857955,0.886364,0.914773,0.831841,0.789773,0.830559,0.842296,0.851972
5,Hybrid reranker alpha05 top-10,dev,176,0.857955,0.926136,0.937500,0.943182,0.857955,0.926136,0.937500,0.943182,0.892045,0.857955,0.897997,0.902891,0.904915


Saved metrics: /content/retrieval_evaluation/dev/retrieval_metrics_dev.csv
Saved per-claim results: /content/retrieval_evaluation/dev/retrieval_per_claim_dev.csv


In [5]:
print("metrics_df type:", type(metrics_df))
print("metrics_df shape:", metrics_df.shape)
print("metrics_df columns:", metrics_df.columns.tolist())
display(metrics_df.head())

metrics_df type: <class 'pandas.core.frame.DataFrame'>
metrics_df shape: (6, 16)
metrics_df columns: ['method', 'mode', 'num_claims', 'Recall@1', 'Recall@3', 'Recall@5', 'Recall@10', 'Hit@1', 'Hit@3', 'Hit@5', 'Hit@10', 'MRR@10', 'nDCG@1', 'nDCG@3', 'nDCG@5', 'nDCG@10']


,method,mode,num_claims,Recall@1,Recall@3,Recall@5,Recall@10,Hit@1,Hit@3,Hit@5,Hit@10,MRR@10,nDCG@1,nDCG@3,nDCG@5,nDCG@10
0,BM25 top-20,dev,176,0.732955,0.812500,0.840909,0.863636,0.732955,0.812500,0.840909,0.863636,0.777264,0.732955,0.779423,0.790911,0.798249
1,Dense top-20,dev,176,0.670455,0.767045,0.818182,0.880682,0.670455,0.767045,0.818182,0.880682,0.729796,0.670455,0.724701,0.745231,0.765599
2,Hybrid alpha03 top-20,dev,176,0.767045,0.840909,0.875000,0.875000,0.767045,0.840909,0.875000,0.875000,0.809375,0.767045,0.811416,0.825850,0.825850
3,Hybrid alpha05 top-20,dev,176,0.772727,0.863636,0.903409,0.926136,0.772727,0.863636,0.903409,0.926136,0.825110,0.772727,0.825621,0.842252,0.849803
4,Hybrid alpha07 top-20,dev,176,0.789773,0.857955,0.886364,0.914773,0.789773,0.857955,0.886364,0.914773,0.831841,0.789773,0.830559,0.842296,0.851972


# Select the best hybrid alpha on dev

In [6]:
if MODE == "dev":
    alpha_rows = metrics_df[metrics_df["method"].str.match(r"Hybrid alpha\d+ top-20", na=False)].copy()

    if len(alpha_rows) > 0:
        alpha_rows = alpha_rows.sort_values(
            by=["MRR@10", "Recall@5", "Recall@10", "nDCG@10"],
            ascending=False,
        )

        best_row = alpha_rows.iloc[0].to_dict()
        match = re.search(r"(alpha\d+)", best_row["method"])
        selected_alpha_tag = match.group(1)
        selected_alpha = int(selected_alpha_tag.replace("alpha", "")) / 10.0

        selection = {
            "mode": MODE,
            "selected_alpha": selected_alpha,
            "selected_alpha_tag": selected_alpha_tag,
            "selection_priority": ["MRR@10", "Recall@5", "Recall@10", "nDCG@10"],
            "selected_method": best_row["method"],
            "selected_metrics": best_row,
        }

        with open(OUTPUT_DIR / "selected_alpha_dev.json", "w", encoding="utf-8") as f:
            json.dump(selection, f, ensure_ascii=False, indent=2)

        alpha_rows.to_csv(OUTPUT_DIR / "alpha_selection_metrics_dev.csv", index=False)

        display(alpha_rows)

        print(f"Selected alpha: {selected_alpha}")
        print(f"Selected alpha tag: {selected_alpha_tag}")
        print("Set RERANK_ALPHA to this value in the retrieval output notebook before running reranker for dev and test.")
    else:
        print("No hybrid alpha top-20 files found for alpha selection.")
else:
    print("Alpha selection is only performed on dev.")

,method,mode,num_claims,Recall@1,Recall@3,Recall@5,Recall@10,Hit@1,Hit@3,Hit@5,Hit@10,MRR@10,nDCG@1,nDCG@3,nDCG@5,nDCG@10
4,Hybrid alpha07 top-20,dev,176,0.789773,0.857955,0.886364,0.914773,0.789773,0.857955,0.886364,0.914773,0.831841,0.789773,0.830559,0.842296,0.851972
3,Hybrid alpha05 top-20,dev,176,0.772727,0.863636,0.903409,0.926136,0.772727,0.863636,0.903409,0.926136,0.825110,0.772727,0.825621,0.842252,0.849803
2,Hybrid alpha03 top-20,dev,176,0.767045,0.840909,0.875000,0.875000,0.767045,0.840909,0.875000,0.875000,0.809375,0.767045,0.811416,0.825850,0.825850


Selected alpha: 0.7
Selected alpha tag: alpha07
Set RERANK_ALPHA to this value in the retrieval output notebook before running reranker for dev and test.


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Compact final table

In [8]:
best_tag = alpha_tag(BEST_ALPHA_FROM_DEV)

compact_methods = [
    "BM25 top-20",
    "Dense top-20",
    f"Hybrid {best_tag} top-20",
    f"Hybrid reranker {best_tag} top-10",
]

compact_df = metrics_df[metrics_df["method"].isin(compact_methods)].copy()

compact_path = OUTPUT_DIR / f"retrieval_metrics_compact_{MODE}.csv"
compact_df.to_csv(compact_path, index=False)

display(compact_df)

print(f"Saved compact table: {compact_path}")

,method,mode,num_claims,Recall@1,Recall@3,Recall@5,Recall@10,Hit@1,Hit@3,Hit@5,Hit@10,MRR@10,nDCG@1,nDCG@3,nDCG@5,nDCG@10
0,BM25 top-20,dev,176,0.732955,0.812500,0.840909,0.863636,0.732955,0.812500,0.840909,0.863636,0.777264,0.732955,0.779423,0.790911,0.798249
1,Dense top-20,dev,176,0.670455,0.767045,0.818182,0.880682,0.670455,0.767045,0.818182,0.880682,0.729796,0.670455,0.724701,0.745231,0.765599
3,Hybrid alpha05 top-20,dev,176,0.772727,0.863636,0.903409,0.926136,0.772727,0.863636,0.903409,0.926136,0.825110,0.772727,0.825621,0.842252,0.849803
5,Hybrid reranker alpha05 top-10,dev,176,0.857955,0.926136,0.937500,0.943182,0.857955,0.926136,0.937500,0.943182,0.892045,0.857955,0.897997,0.902891,0.904915


Saved compact table: /content/retrieval_evaluation/dev/retrieval_metrics_compact_dev.csv
